[![OpenSD2026 - Belgium](../../../assets/OpenSD_bannerlogo.jpg)](https://www.vub.be/en/event/opensd-summer-school-belgium)

Copyright © 2026 OpenSD2026 contributors. All rights reserved. The materials in this repository are provided for educational use only. See [LICENSE](../../../LICENSE).

# Test-Driven Development with `pytest`

**Test-Driven Development (TDD)** is an approach where we write the **test before writing the implementation**.

The workflow is simple:

1. **Write a test** describing what the code should do.
2. **Run the test** and see it fail.
3. **Write the code** needed to make the test pass.
4. **Refactor** while keeping the tests passing.

## Why is this especially relevant in the era of AI?

AI tools can generate code very quickly, but generated code is not necessarily **correct**.

This makes testing even more important. Instead of asking an AI to generate some code and then trying to understand whether it works, we can first define the **expected behaviour through tests**.

The tests then become a clear specification that both humans and AI-generated code must satisfy:

> **We define what "correct" means first, then generate or write the implementation.**

For scientific and engineering software, this is particularly useful because we often already know physical properties, analytical solutions, limiting cases, or expected outputs that can be turned into tests.

In this notebook, we will apply this idea to a simple **projectile-motion model**: first define the expected physical behaviour, then implement the model that satisfies it.

## Projectile motion

Suppose we need a function called `landing_distance(speed, angle_degrees)` that returns where a ball lands. We do not know the equation yet, but we found the following examples in [OpenStax University Physics, Figure 4.15](https://openstax.org/books/university-physics-volume-1/pages/4-3-projectile-motion):

| Speed [m/s] | Angle | Range [m] |
| ---: | ---: | ---: |
| 30 | 45° | 91.8 |
| 40 | 45° | 163 |
| 50 | 45° | 255 |
| 50 | 15° | 128 |
| 50 | 75° | 128 |

These values assume level ground and no air resistance. They are rounded, so we will allow a 1% difference in the tests.

## Write the test first

Start with one value that we already know.

In [8]:
import pytest


def test_one_reference_value():
    result = landing_distance(speed=30, angle_degrees=45)
    assert result == pytest.approx(91.8, rel=0.01)

`landing_distance` has not been written yet. If this test is run now, it fails. This is the **red** step.

## Write the function

We can now implement the projectile equation and run the same test again.

In [10]:
from math import radians, sin


def landing_distance(speed, angle_degrees, gravity=9.81):
    angle = radians(angle_degrees)
    return speed**2 * sin(2 * angle) / gravity

In [11]:
test_one_reference_value()

No exception means that the test passed. This is the **green** step.

## Test the other known values

Instead of copying the test five times, use `@pytest.mark.parametrize`. Pytest will report each row as a separate test.

In [14]:
REFERENCE_CASES = [
    (30, 45, 91.8),
    (40, 45, 163),
    (50, 45, 255),
    (50, 15, 128),
    (50, 75, 128),
]


@pytest.mark.parametrize(
    ("speed", "angle_degrees", "expected"),
    REFERENCE_CASES,
)
def test_openstax_reference(speed, angle_degrees, expected):
    result = landing_distance(speed, angle_degrees)
    assert result == pytest.approx(expected, rel=0.01)

> **Exercise:** We also know that a vertical launch ($90°$) has no horizontal motion. Add a separate test for this case. Use an absolute tolerance because the expected result is zero.

<details>
<summary><strong>Show answer</strong></summary>

```python
def test_vertical_launch_has_zero_range():
    result = landing_distance(speed=10, angle_degrees=90)
    assert result == pytest.approx(0, abs=1e-12)
```

</details>

## 3. Add more behaviour with parametrization

Now we expand the specification with the remaining reference values. The `@pytest.mark.parametrize` decorator passes each tuple in `REFERENCE_CASES` to the same test function, so pytest reports every case separately.

We also add the known $90°$ limiting case. The loop at the bottom executes the cases in this notebook; pytest performs that expansion automatically in a test file.

## Run the tests with pytest

The relevant files in the finished example are:

```text
012_test_driven_dev/
├── Awesome_openSD/
│   └── projectile.py
├── tests/
│   └── test_projectile.py
└── 012_test_driven_dev.ipynb
```

From the repository root, run:

```bash
uv run pytest
or
uv run pytest notebooks/WS1_An_intro_to_python/012_test_driven_dev/tests/test_projectile.py -q
```

Moving the function into `projectile.py` is the useful refactor here. The tests stay the same and can now be run automatically.